# 🧠 강화학습 6강 — PPO: 행동을 직접 고르는 두뇌
### 🚗 뒤집힌 로드스터, 이번엔 착지한다 · 버튼 4개 → 🕹 조이스틱

**AI CITY BUILDERS · Connect AI LAB**

5강의 끝에서 DQN 은 뒤집힌 로드스터 앞에서 멈췄습니다. 반짝 배웠다가 잊고, 마지막 두뇌는 **0%**.
오늘은 **같은 세상**에 **다른 두뇌**를 붙입니다. 점수표(Q)를 외워 고르는 두뇌가 아니라, **행동을 고를 확률을 직접 배우는 두뇌 — PPO** 입니다.

| 지난 강의 | 오늘 PPO 에서 |
|---|---|
| **2강 탐험 ε** | ε 동전이 사라집니다. 확률대로 고르는 것 자체가 탐험입니다 |
| **3강 기억 상자** | 오래된 기억을 쌓지 않습니다. 방금 해 본 판들로 배우고 버립니다 |
| **4강 정책 π(a\|s) · γ** | 4강의 「상황마다 행동을 고를 확률」을 **직접** 배웁니다 · γ 는 그대로 |
| **5강 DQN · 로드스터** | DQN 이 0% 였던 그 세상을 그대로 풉니다 |

| 오늘의 이야기 | 셀 | 하는 일 |
|---|---|---|
| ① 지난 이야기 | 01 ~ 02 | 뒤집힌 로드스터 다시 열기 · 무작위는 0번 착지 |
| ② 점수표 vs 확률 | 03 | DQN 과 PPO 는 같은 상태를 다르게 본다 |
| ③ PPO 네 조각 | 04 ~ 07 | 정책 두뇌 · 한 판 통째로 · 얼마나 잘했나 · 조금만 고치기 |
| ④ 착지 | 08 ~ 11 | 네 조각을 합쳐 버튼 4개로 착지 → 조이스틱으로 착지 · 두뇌 속 확률 |
| ⑤ DQN vs PPO | 12 | 같은 세상 · 씨앗 5개 · 실제 결과 |
| 🏆 도전 · 🤗 공개 | 13 ~ 16 | 숫자를 바꿔 착지율 올리기 · 허깅페이스에 PPO 두뇌 올리기 |
| 👉 7강 예고 | 17 | 관절이 여럿인 로봇 — MuJoCo 두 발 걷기 |

> ▶ **런타임 → 모두 실행** (GPU 필요 없음 · 약 10 ~ 15분). 허깅페이스 업로드(15번 셀)는 토큰을 넣어야 진행됩니다.
> 셀마다 움직이며 이해하는 해설 페이지: [aicitybuilders.com/ppo](https://www.aicitybuilders.com/ppo)

## 🔍 PPO 가 뭐예요?

**PPO = Proximal Policy Optimization = 「정책을, 가까이에서만, 조금씩 고치기」**

| 글자 | 뜻 |
|---|---|
| **P** · Proximal | 가까운 — 지금 정책에서 **너무 멀리 가지 않게** |
| **P** · Policy | 정책 π(a\|s) — 4강의 「이 상황에서 이 행동을 고를 확률」 |
| **O** · Optimization | 최적화 — 누적 보상이 커지는 쪽으로 고친다 |

DQN 은 「행동마다 몇 점인지(Q)」 를 외운 다음, 가장 큰 걸 골랐습니다. **정책은 Q 에서 저절로 나오는 부산물**이었죠.
PPO 는 처음부터 **정책 자체를 신경망으로** 둡니다. 상태를 넣으면 「쉬기 10% · 왼쪽 70% · 아래 15% · 오른쪽 5%」 처럼 **확률**이 나옵니다.

**PPO 가 배우는 한 바퀴**
1. **정책 두뇌** — 상태 → 행동 확률 (+ 이 상태가 얼마나 좋은지 예상하는 가치 머리)
2. **한 판 통째로** — 여러 세상을 동시에 돌리며 확률대로 고르고 기록 (ε 없음)
3. **얼마나 잘했나** — 어드밴티지 = 실제로 받은 것 − 예상한 것 (γ 는 4강 그대로)
4. **조금만 고치기** — 잘한 행동의 확률은 올리고, 못한 건 내리되 **한 번에 ±20% 까지만** (clip)

## CELL 01 · 준비

In [ ]:
!pip install -q "gymnasium>=1.0" imageio-ffmpeg huggingface_hub
!apt-get -qq install -y fonts-nanum > /dev/null 2>&1

import os, math, time, json, random, base64, collections, copy, warnings
warnings.filterwarnings('ignore')   # 환경 점검 경고 문구 숨기기 (결과에는 영향 없음)
import numpy as np, torch, torch.nn as nn
import gymnasium as gym, imageio
import matplotlib, matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont
from IPython.display import HTML, display

글꼴경로 = next((p for p in ['/usr/share/fonts/truetype/nanum/NanumGothicBold.ttf',
                            '/System/Library/Fonts/Supplemental/AppleGothic.ttf'] if os.path.exists(p)), None)
if 글꼴경로:
    matplotlib.font_manager.fontManager.addfont(글꼴경로)
    plt.rcParams['font.family'] = matplotlib.font_manager.FontProperties(fname=글꼴경로).get_name()
plt.rcParams['axes.unicode_minus'] = False
def 글꼴(n):
    try: return ImageFont.truetype(글꼴경로, n)
    except Exception: return ImageFont.load_default()

torch.set_num_threads(1)   # 작은 두뇌는 CPU 한 줄이 가장 빠르다
def 영상보기(화면들, 파일, fps=30, 너비=480):
    imageio.mimsave(파일, 화면들, fps=fps, macro_block_size=1)
    display(HTML(f'<video width="{너비}" controls autoplay loop muted><source src="data:video/mp4;base64,'
                 f'{base64.b64encode(open(파일, "rb").read()).decode()}" type="video/mp4"></video>'))
print('✅ 준비 완료 · torch', torch.__version__, '· gymnasium', gym.__version__)

## CELL 02 · 지난 이야기 — 뒤집힌 로드스터

5강 13번 셀에서 직접 만든 세상을 다시 엽니다. 오늘은 한 가지를 더 넣었습니다 — **`로드스터연속`**: 버튼 4개 대신 **아래 추진기 세기(0 ~ 1)와 회전 세기(−1 ~ 1)** 를 직접 고르는 조이스틱 버전입니다.

| | 버튼 (`로드스터`) | 조이스틱 (`로드스터연속`) |
|---|---|---|
| 행동 | 0 쉬기 · 1 왼쪽 냉가스 · 2 아래 추진기 · 3 오른쪽 냉가스 | [아래 추진기 세기, 회전 세기] |
| 행동 공간 | `Discrete(4)` | `Box(2)` — 연속된 숫자 두 개 |
| 연료 | 켜면 정해진 만큼 | 세기에 비례 |

상태 7개(위치 · 속도 · 기울기 sin · cos · 회전 속도), 보상 규칙(착지 +100 · 추락 −100 · 연료 · 다듬기)은 5강과 똑같습니다.

In [ ]:
%%writefile 로드스터_시뮬레이터.py
# 뒤집힌 로드스터 착지 시뮬레이터 — Gymnasium 틀 (Connect AI LAB · 강화학습 5강)
import math, numpy as np, gymnasium as gym
from gymnasium import spaces

class 로드스터(gym.Env):
    """뒤집힌 채로 떨어지는 차를 냉가스 추진기로 돌려 바퀴부터 착지시키기."""
    metadata = {'render_modes': ['rgb_array']}
    G, DT, MAIN, SIDE, MAXT = 1.0, 0.05, 2.2, 1.8, 400            # 중력 · 한 걸음 시간 · 아래 추진기 힘 · 냉가스 힘 · 최대 걸음
    def __init__(self, 시작각=math.pi, 보상=None):
        self.시작각 = 시작각
        self.P = dict(착지=100.0, 추락=-100.0, 연료_가운데=0.3, 연료_옆=0.03, 다듬기=1.0); self.P.update(보상 or {})
        self.action_space = spaces.Discrete(4)                    # 0 쉬기 · 1 왼쪽 냉가스 · 2 아래 추진기 · 3 오른쪽 냉가스
        self.observation_space = spaces.Box(-np.inf, np.inf, (7,), np.float32)
    def _obs(self):
        x, y, vx, vy, th, w = self.s
        return np.array([x / 5, y / 10, vx / 2, vy / 2, math.sin(th), math.cos(th), w / 2], np.float32)
    def _모양(self):                                              # 착지대 · 느림 · 똑바름에 가까울수록 높은 점수
        x, y, vx, vy, th, w = self.s; thn = (th + math.pi) % (2 * math.pi) - math.pi
        return -10 * math.hypot(x / 5, y / 10) - 10 * math.hypot(vx / 2, vy / 2) - 10 * abs(thn)
    def reset(self, seed=None, options=None):
        super().reset(seed=seed); r = self.np_random
        self.s = np.array([r.uniform(-2.5, 2.5), 10.0, r.uniform(-0.8, 0.8), 0.0, math.pi, r.uniform(-0.3, 0.3)], np.float64)
        self.s[4] = self.시작각 * (1 if r.random() < 0.5 else -1) + r.uniform(-0.15, 0.15)   # 뒤집힌 채로 (왼쪽 · 오른쪽 어느 쪽으로든)
        self.t = 0; self.prev = self._모양(); return self._obs(), {}
    def step(self, a):
        x, y, vx, vy, th, w = self.s; P = self.P; r = 0.0
        ax, ay, aw = 0.0, -self.G, 0.0
        if a == 2: ax += -math.sin(th) * self.MAIN; ay += math.cos(th) * self.MAIN; r -= P['연료_가운데']   # 차 바닥 쪽으로 미는 힘
        elif a == 1: aw -= self.SIDE; r -= P['연료_옆']
        elif a == 3: aw += self.SIDE; r -= P['연료_옆']
        vx += ax * self.DT; vy += ay * self.DT; w += aw * self.DT; w *= 0.96
        x += vx * self.DT; y += vy * self.DT; th += w * self.DT
        self.s = np.array([x, y, vx, vy, th, w]); self.t += 1
        sh = self._모양(); r += P['다듬기'] * (sh - self.prev); self.prev = sh
        끝, 결과 = False, ''
        thn = (th + math.pi) % (2 * math.pi) - math.pi
        if y <= 0:
            끝 = True
            if abs(x) <= 1.5 and abs(thn) < 0.35 and vy > -1.5 and abs(vx) < 1.2: r += P['착지']; 결과 = '착지'
            else: r += P['추락']; 결과 = '추락'
        elif abs(x) > 8 or y > 16: 끝 = True; r += P['추락']; 결과 = '밖으로'
        잘림 = self.t >= self.MAXT and not 끝
        return self._obs(), r, 끝, 잘림, {'결과': 결과 or ('시간 끝' if 잘림 else '')}

class 로드스터연속(로드스터):
    """6강 · 버튼이 아니라 조이스틱 — 아래 추진기 세기(0~1)와 회전 세기(-1~1)를 직접 고른다."""
    def __init__(self, 시작각=math.pi, 보상=None):
        super().__init__(시작각, 보상)
        self.action_space = spaces.Box(np.array([0.0, -1.0], np.float32), np.array([1.0, 1.0], np.float32))
    def step(self, a):
        세기 = float(np.clip(a[0], 0, 1)); 회전 = float(np.clip(a[1], -1, 1))
        x, y, vx, vy, th, w = self.s; P = self.P; r = 0.0
        ax = -math.sin(th) * self.MAIN * 세기; ay = -self.G + math.cos(th) * self.MAIN * 세기; aw = self.SIDE * 회전
        r -= P['연료_가운데'] * 세기 + P['연료_옆'] * abs(회전)
        vx += ax * self.DT; vy += ay * self.DT; w += aw * self.DT; w *= 0.96
        x += vx * self.DT; y += vy * self.DT; th += w * self.DT
        self.s = np.array([x, y, vx, vy, th, w]); self.t += 1
        sh = self._모양(); r += P['다듬기'] * (sh - self.prev); self.prev = sh
        끝, 결과 = False, ''
        thn = (th + math.pi) % (2 * math.pi) - math.pi
        if y <= 0:
            끝 = True
            if abs(x) <= 1.5 and abs(thn) < 0.35 and vy > -1.5 and abs(vx) < 1.2: r += P['착지']; 결과 = '착지'
            else: r += P['추락']; 결과 = '추락'
        elif abs(x) > 8 or y > 16: 끝 = True; r += P['추락']; 결과 = '밖으로'
        잘림 = self.t >= self.MAXT and not 끝
        return self._obs(), r, 끝, 잘림, {'결과': 결과 or ('시간 끝' if 잘림 else '')}

In [ ]:
from 로드스터_시뮬레이터 import 로드스터, 로드스터연속
from gymnasium.utils.env_checker import check_env
check_env(로드스터()); check_env(로드스터연속())
print('✅ 두 세상 모두 check_env 통과 — 버튼 Discrete(4) · 조이스틱 Box(2)')

def 로드스터그림(e, a=None, 제목='', 색=(0, 255, 65), 세기=None):
    W, H, sc = 480, 360, 26; ox, gy = W / 2, H - 40
    im = Image.new('RGB', (W, H), (7, 8, 10)); d = ImageDraw.Draw(im)
    d.rectangle([0, gy, W, H], fill=(20, 23, 28)); d.rectangle([ox - 1.5 * sc, gy - 3, ox + 1.5 * sc, gy + 3], fill=(255, 181, 71))
    x, y, vx, vy, th, w = e.s; cx, cy = ox + x * sc, gy - y * sc
    def rot(px, py): c, s = math.cos(th), math.sin(th); return (cx + px * c + py * s, cy - (-px * s + py * c))
    d.polygon([rot(-16, -5), rot(16, -5), rot(16, 3), rot(6, 9), rot(-8, 9), rot(-16, 3)], fill=(225, 40, 50))
    d.polygon([rot(-5, 8), rot(5, 8), rot(2, 12), rot(-3, 12)], fill=(191, 239, 255))
    for wx in (-10, 10): p = rot(wx, -7); d.ellipse([p[0] - 5, p[1] - 5, p[0] + 5, p[1] + 5], fill=(30, 30, 30), outline=(160, 160, 160))
    아래 = 세기[0] if 세기 is not None else (1.0 if a == 2 else 0.0)
    회전 = 세기[1] if 세기 is not None else (-1.0 if a == 1 else 1.0 if a == 3 else 0.0)
    if 아래 > 0.05: d.polygon([rot(-5, -9), rot(5, -9), rot(0, -9 - 20 * 아래)], fill=(255, 181, 71))
    if 회전 < -0.05: d.polygon([rot(16, 2), rot(16, -3), rot(16 + 14 * -회전, 0)], fill=(191, 239, 255))
    if 회전 > 0.05: d.polygon([rot(-16, 2), rot(-16, -3), rot(-16 - 14 * 회전, 0)], fill=(191, 239, 255))
    thn = math.degrees((th + math.pi) % (2 * math.pi) - math.pi)
    d.text((12, 10), 제목, fill=색, font=글꼴(18)); d.text((12, 36), f'높이 {y:4.1f} · 세로속도 {vy:+.2f} · 기울기 {thn:+5.0f}°', fill=(200, 210, 220), font=글꼴(13))
    return np.array(im)

e = 로드스터(); 결과 = collections.Counter()
for 씨 in range(200):
    o, _ = e.reset(seed=씨); 끝 = 잘림 = False
    while not (끝 or 잘림): o, r, 끝, 잘림, 정보 = e.step(e.action_space.sample())
    결과[정보['결과']] += 1
print('🎲 무작위로 200판 →', dict(결과), '— 착지 0번. 5강 DQN 의 마지막 두뇌도 0% 였습니다.')
display(HTML('<video width="480" autoplay loop muted playsinline src="https://www.aicitybuilders.com/brain/rd_dqn.mp4"></video>'
             '<div style="color:#ff5d5d;font-weight:800">5강 · DQN 마지막 두뇌 — 착지 못 함</div>'))

## CELL 03 · 점수표 vs 확률 — 같은 상태, 다른 대답

같은 순간(뒤집힌 채 떨어지는 중)을 두 두뇌에 보여 줍니다.

- **DQN 두뇌**: 「쉬기 +3.1점 · 왼쪽 +5.8점 · …」 → **가장 큰 점수 하나**를 고릅니다. 탐험은 ε 동전으로 따로 합니다(2강).
- **PPO 두뇌**: 「쉬기 12% · 왼쪽 61% · …」 → **확률대로 굴려서** 고릅니다. 61% 짜리가 자주 나오지만, 12% 짜리도 가끔 나옵니다. **그게 곧 탐험**입니다.

아직 아무것도 배우지 않은 PPO 두뇌는 네 행동을 거의 똑같이(각 25%) 봅니다. 배울수록 잘한 행동 쪽으로 확률이 모입니다.

In [ ]:
class 정책두뇌(nn.Module):
    """두 머리: 정책(행동) · 가치(이 상태가 얼마나 좋은가)."""
    def __init__(self, 상태수, 행동수, 연속=False, 은닉=64):
        super().__init__()
        self.연속 = 연속
        몸 = lambda 출력: nn.Sequential(nn.Linear(상태수, 은닉), nn.Tanh(), nn.Linear(은닉, 은닉), nn.Tanh(), nn.Linear(은닉, 출력))
        self.정책 = 몸(행동수)
        self.가치 = 몸(1)
        if 연속:
            self.흔들림 = nn.Parameter(torch.full((행동수,), -0.5))   # log 표준편차 — 처음엔 넓게 탐험, 배울수록 좁아진다
        with torch.no_grad():                                       # 처음엔 모든 행동을 비슷하게 (작은 출력)
            self.정책[-1].weight.mul_(0.01); self.정책[-1].bias.zero_()

    def 분포(self, s):
        out = self.정책(s)
        if self.연속:
            return torch.distributions.Normal(out, self.흔들림.exp().expand_as(out))
        return torch.distributions.Categorical(logits=out)

    def 고르기(self, s):
        d = self.분포(s); a = d.sample()
        lp = d.log_prob(a).sum(-1) if self.연속 else d.log_prob(a)
        return a, lp, self.가치(s).squeeze(-1)


e = 로드스터(); o, _ = e.reset(seed=3)
torch.manual_seed(0); 뇌 = 정책두뇌(7, 4)
with torch.no_grad(): 확률 = 뇌.분포(torch.as_tensor(o)).probs.numpy()
버튼 = ['쉬기', '왼쪽 냉가스', '아래 추진기', '오른쪽 냉가스']
for 이름, p in zip(버튼, 확률): print(f'  {이름:<7} {p*100:5.1f}%  ' + '█' * int(p * 60))
print('\n같은 상태에서 20번 골라 보면 →', collections.Counter(버튼[int(뇌.분포(torch.as_tensor(o)).sample())] for _ in range(20)))
print('배우기 전이라 거의 고르게 나옵니다. 이 「골고루」가 곧 탐험 — ε 동전이 필요 없습니다.')

## CELL 04 · PPO 조각 ① 정책 두뇌 — 두 개의 머리

위 칸에서 쓴 `정책두뇌` 의 속을 봅니다. 몸통은 5강 두뇌처럼 작은 신경망이고, **머리가 두 개**입니다.

| 머리 | 대답 | 쓰는 곳 |
|---|---|---|
| **정책** | 행동마다 고를 확률 (버튼) · 세기의 평균과 흔들림 (조이스틱) | 행동 고르기 |
| **가치** | 「이 상태가 앞으로 얼마나 좋을까」 숫자 하나 — V(s) | ③ 얼마나 잘했나 |

> 조이스틱은 확률을 「칸」으로 나눌 수 없습니다. 대신 **평균 세기**와 **흔들림(표준편차)** 을 냅니다. 평균 0.7 · 흔들림 0.3 이면 0.4 ~ 1.0 근처에서 굴려 고릅니다. 배울수록 흔들림이 줄어 「정확한 세기」를 내게 됩니다.

In [ ]:
print(뇌)
print('\n정책 머리 숫자 :', sum(p.numel() for p in 뇌.정책.parameters()), '개 · 가치 머리 숫자 :', sum(p.numel() for p in 뇌.가치.parameters()), '개')
with torch.no_grad(): print('이 상태의 가치 V(s) 예상 :', round(float(뇌.가치(torch.as_tensor(o))), 3), '(아직 아무것도 모름)')

조이 = 정책두뇌(7, 2, 연속=True)
with torch.no_grad(): d = 조이.분포(torch.as_tensor(o))
print('\n🕹 조이스틱 두뇌 — 평균 세기', np.round(d.mean.numpy(), 3), '· 흔들림', np.round(d.stddev.numpy(), 3))
print('   굴려 본 세기 3번 :', [np.round(np.clip(d.sample().numpy(), [0, -1], [1, 1]), 2).tolist() for _ in range(3)])

## CELL 05 · PPO 조각 ② 한 판 통째로 — 여러 세상을 동시에

DQN 은 한 걸음 할 때마다 기억 상자에 넣고 꺼내 복습했습니다(3강). PPO 는 다르게 합니다.

1. 지금 정책 그대로 **여러 세상(여기선 8개)** 을 동시에 돌립니다.
2. 세상마다 256 걸음씩, 총 **2,048 걸음**을 확률대로 고르며 기록합니다 — 상태 · 행동 · 그때의 확률 · 보상 · 가치 예상.
3. 이 묶음으로 배운 뒤에는 **버립니다.** 다음엔 고쳐진 정책으로 새로 해 봅니다.

> 오래된 기억이 없으니, 5강에서 본 「드문 착지 경험이 추락 기억에 묻혀 잊히는」 일이 덜 생깁니다. 대신 경험을 많이 모아야 해서 **여러 세상을 동시에** 돌립니다. 세상이 많을수록 빨리 배웁니다 — 컴퓨터 여러 대로 시뮬레이터를 돌리는 이유입니다.

In [ ]:
세상들 = [로드스터() for _ in range(8)]
s = np.stack([w.reset(seed=i)[0] for i, w in enumerate(세상들)])
기록 = collections.Counter(); 판 = 0
for t in range(256):
    with torch.no_grad(): a, lp, v = 뇌.고르기(torch.as_tensor(s, dtype=torch.float32))
    for i, w in enumerate(세상들):
        o2, r, 끝, 잘림, 정보 = w.step(int(a[i])); 기록[버튼[int(a[i])]] += 1
        if 끝 or 잘림: 판 += 1; o2, _ = w.reset(seed=1000 + 판)
        s[i] = o2
print(f'세상 8개 × 256 걸음 = {sum(기록.values()):,} 걸음을 한 번에 모았습니다 · 끝난 판 {판}개')
print('고른 행동 :', dict(기록), '← ε 없이도 네 버튼을 골고루 눌러 봅니다')

## CELL 06 · PPO 조각 ③ 얼마나 잘했나 — 어드밴티지

행동 하나를 칭찬할지 혼낼지는 **「예상보다 잘됐나」** 로 정합니다.

**어드밴티지 A = (실제로 받은 보상 + γ × 다음 상태의 가치) − 이 상태의 가치 예상**

- A > 0 → 예상보다 잘됐다 → 그 행동의 확률을 **올린다**
- A < 0 → 예상보다 못했다 → 그 행동의 확률을 **내린다**

γ(감마)는 4강의 할인율 그대로 0.99 입니다. 실제 코드는 이 차이를 몇 걸음 뒤까지 이어 붙여 부드럽게 만드는 **GAE** 를 씁니다(λ = 0.95).

> 사람으로 치면: 「오늘 발표 70점」 자체보다 「평소 50점이던 내가 70점」 이 중요합니다. 평소(가치 예상)보다 나았던 행동을 더 하게 됩니다.

In [ ]:
γ, λ = 0.99, 0.95
보상 = np.array([-0.3, -0.3, 0.8, 1.2, -0.03, 100.0]); 가치 = np.array([5.0, 6.0, 9.0, 20.0, 105.0, 90.0]); 다음가치 = 0.0
어드 = np.zeros_like(보상); 누적 = 0.0
for t in reversed(range(len(보상))):
    nv = 다음가치 if t == len(보상) - 1 else 가치[t + 1]
    델타 = 보상[t] + γ * nv - 가치[t]; 누적 = 델타 + γ * λ * 누적; 어드[t] = 누적
for t in range(len(보상)):
    print(f'  걸음 {t+1} · 보상 {보상[t]:+7.2f} · 예상 V {가치[t]:5.1f} → 어드밴티지 {어드[t]:+7.2f}  ' + ('▲ 확률 올림' if 어드[t] > 0 else '▼ 확률 내림'))
print('\n착지로 이어진 행동들은 + 를 받습니다 — 착지(+100)가 뒤에서부터 앞으로 번져 오기 때문(4강의 γ).')
print('5번 걸음은 − : 「거의 다 왔다(예상 105)」고 너무 믿었는데 실제론 그만큼 못 받았으니, 그때 한 행동의 확률은 내려갑니다.')

## CELL 07 · PPO 조각 ④ 조금만 고치기 — clip

이제 확률을 고칩니다. 그런데 **한 번에 너무 많이** 고치면 어떻게 될까요? 운 좋게 한 번 착지한 행동의 확률을 10% → 90% 로 올려 버리면, 그 행동만 하다가 계속 추락할 수 있습니다. 5강 DQN 이 「반짝 배웠다 잊는」 것과 비슷한 흔들림입니다.

PPO 의 핵심 장치:

**비율 = 새 정책의 확률 ÷ 옛 정책의 확률** 을 **0.8 ~ 1.2 (1 ± 0.2)** 밖으로는 쳐 주지 않는다.

| | 어드밴티지 > 0 (잘했다) | 어드밴티지 < 0 (못했다) |
|---|---|---|
| 하고 싶은 것 | 그 행동 확률 올리기 | 그 행동 확률 내리기 |
| clip | **1.2 배** 넘게 올려도 더 칭찬 안 함 | **0.8 배** 밑으로 내려도 더 혼내지 않음 |

> 「가까이에서만(Proximal) 정책을(Policy) 고친다(Optimization)」 — 이름 그대로입니다. 한 번에 조금씩, 그러나 같은 묶음으로 10번 반복해서 꾸준히.

In [ ]:
비율 = np.linspace(0.5, 1.6, 200)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.3))
for k, A in enumerate([+1.0, -1.0]):
    목표 = np.minimum(비율 * A, np.clip(비율, 0.8, 1.2) * A)
    ax[k].plot(비율, 비율 * A, '--', c='#9aa3ad', label='그냥 고치기 (끝없이)')
    ax[k].plot(비율, 목표, lw=3, c='#00c832', label='PPO clip')
    ax[k].axvspan(0.8, 1.2, color='#bfefff', alpha=.35); ax[k].axvline(1, c='#666', lw=.8)
    ax[k].set_title('잘했다 (A > 0) → 1.2 배에서 멈춤' if A > 0 else '못했다 (A < 0) → 0.8 배에서 멈춤')
    ax[k].set_xlabel('새 확률 ÷ 옛 확률'); ax[k].set_ylabel('고치는 힘'); ax[k].legend(fontsize=9)
plt.tight_layout(); plt.show()
print('하늘색 띠(0.8 ~ 1.2) 밖에서는 초록 선이 평평 — 더 멀리 가도 이득이 없으니, 두뇌가 한 번에 크게 바뀌지 않습니다.')

## CELL 08 · 네 조각을 합치면 — PPO 학습 함수

CELL 04 ~ 07 의 네 조각을 한 함수에 넣었습니다. 코드 안의 ② ③ ④ 표시를 따라가 보세요.
이 함수 하나로 버튼(로드스터)도, 조이스틱(로드스터연속)도 풉니다 — 세상의 행동 공간을 보고 알아서 바꿉니다.

In [ ]:
def PPO학습(세상만들기, 총걸음, 세상수=8, 한번에=256, 감마=0.99, 람다=0.95, 자르기=0.2, 학습률=3e-4,
           반복=10, 묶음=256, 탐험보너스=0.0, 씨앗=0, 보고=100_000, 평가=None, 은닉=64, 기록=None):
    torch.manual_seed(씨앗); np.random.seed(씨앗)
    세상들 = [세상만들기() for _ in range(세상수)]
    연속 = hasattr(세상들[0].action_space, 'shape') and len(세상들[0].action_space.shape or ()) > 0
    상태수 = 세상들[0].observation_space.shape[0]
    행동수 = 세상들[0].action_space.shape[0] if 연속 else 세상들[0].action_space.n
    뇌 = 정책두뇌(상태수, 행동수, 연속, 은닉); opt = torch.optim.Adam(뇌.parameters(), lr=학습률, eps=1e-5)
    s = np.stack([e.reset(seed=씨앗 * 1000 + i)[0] for i, e in enumerate(세상들)]); 판번호 = 세상수
    판점수, 지금점수, 결과기록, 평가기록 = [], np.zeros(세상수), [], []
    걸음 = 0; t0 = time.time(); 다음보고 = 보고; 최고 = (-1, None)
    while 걸음 < 총걸음:
        # ② 한 판 통째로 — 지금 정책 그대로, 확률대로 고르며 기록
        S, A, LP, R, D, V = [], [], [], [], [], []
        for _ in range(한번에):
            st = torch.as_tensor(s, dtype=torch.float32)
            with torch.no_grad(): a, lp, v = 뇌.고르기(st)
            an = a.numpy(); s2 = np.empty_like(s); r = np.zeros(세상수, np.float32); d = np.zeros(세상수, np.float32)
            for i, e in enumerate(세상들):
                act = np.clip(an[i], e.action_space.low, e.action_space.high) if 연속 else int(an[i])
                o, ri, 끝, 잘림, 정보 = e.step(act); 지금점수[i] += ri
                if 끝 or 잘림:
                    판점수.append(지금점수[i]); 지금점수[i] = 0; 결과기록.append(정보.get('결과', ''))
                    o, _ = e.reset(seed=씨앗 * 1000 + 판번호); 판번호 += 1
                    d[i] = 1.0   # 잘림도 끝으로 본다 (단순하게)
                s2[i] = o; r[i] = ri
            S.append(st); A.append(a); LP.append(lp); R.append(torch.as_tensor(r)); D.append(torch.as_tensor(d)); V.append(v)
            s = s2; 걸음 += 세상수
        with torch.no_grad(): 마지막가치 = 뇌.가치(torch.as_tensor(s, dtype=torch.float32)).squeeze(-1)
        # ③ 얼마나 잘했나 — 어드밴티지 (GAE): 실제로 받은 것이 예상보다 얼마나 좋았나
        어드 = torch.zeros(한번에, 세상수); 누적 = torch.zeros(세상수)
        for t in reversed(range(한번에)):
            다음가치 = 마지막가치 if t == 한번에 - 1 else V[t + 1]
            델타 = R[t] + 감마 * 다음가치 * (1 - D[t]) - V[t]
            누적 = 델타 + 감마 * 람다 * (1 - D[t]) * 누적; 어드[t] = 누적
        목표가치 = 어드 + torch.stack(V)
        S_ = torch.cat(S); A_ = torch.cat(A); LP_ = torch.cat(LP); 어드_ = 어드.reshape(-1); 목표_ = 목표가치.reshape(-1)
        # ④ 조금만 고치기 — 새 확률 / 옛 확률 을 1 ± 자르기 안에서만
        n = S_.shape[0]
        for _ in range(반복):
            순서 = torch.randperm(n)
            for k in range(0, n, 묶음):
                i = 순서[k:k + 묶음]
                d_ = 뇌.분포(S_[i])
                새lp = d_.log_prob(A_[i]).sum(-1) if 연속 else d_.log_prob(A_[i])
                비율 = (새lp - LP_[i]).exp()
                a_ = 어드_[i]; a_ = (a_ - a_.mean()) / (a_.std() + 1e-8)
                정책손실 = -torch.min(비율 * a_, 비율.clamp(1 - 자르기, 1 + 자르기) * a_).mean()
                가치손실 = ((뇌.가치(S_[i]).squeeze(-1) - 목표_[i]) ** 2).mean()
                엔트 = d_.entropy().sum(-1).mean() if 연속 else d_.entropy().mean()
                loss = 정책손실 + 0.5 * 가치손실 - 탐험보너스 * 엔트
                opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(뇌.parameters(), 0.5); opt.step()
        if 기록 is not None:
            기록.append(dict(걸음=걸음, 최근=float(np.mean(판점수[-50:])) if 판점수 else 0.0,
                           착지=float(np.mean([x == '착지' for x in 결과기록[-50:]])) if 결과기록 else 0.0))
        if 걸음 >= 다음보고:
            다음보고 += 보고
            최근 = np.mean(판점수[-50:]) if 판점수 else 0
            최근착지 = np.mean([x == '착지' for x in 결과기록[-50:]]) * 100 if 결과기록 else 0
            줄 = f'[{걸음:>9,}걸음 | {time.time()-t0:5.0f}초] 최근 50판 평균 {최근:7.1f} · 착지 {최근착지:3.0f}%'
            if 평가:
                점수 = 평가(뇌); 평가기록.append((걸음, 점수)); 줄 += f' | 시험 {점수:5.1f}'
                if 점수 >= 최고[0]: 최고 = (점수, {k: v.clone() for k, v in 뇌.state_dict().items()}); 줄 += '  🏅'
            print(줄, flush=True)
    return 뇌, 판점수, 평가기록, 최고


def 착지율(뇌, 세상만들기, 판=50, 시작=5000, 확정=True):
    """처음 보는 판에서 — 확정: 가장 확률 높은 행동(버튼) / 평균 세기(조이스틱)."""
    e = 세상만들기(); ok = 0
    for 씨 in range(시작, 시작 + 판):
        o, _ = e.reset(seed=씨); 끝 = 잘림 = False
        while not (끝 or 잘림):
            with torch.no_grad():
                d = 뇌.분포(torch.as_tensor(o, dtype=torch.float32))
                if 뇌.연속: a = np.clip((d.mean if 확정 else d.sample()).numpy(), e.action_space.low, e.action_space.high)
                else: a = int(d.probs.argmax()) if 확정 else int(d.sample())
            o, r, 끝, 잘림, 정보 = e.step(a)
        ok += 정보['결과'] == '착지'
    return ok / 판 * 100


def 곡선(판점수, 제목, 기준=None):
    plt.figure(figsize=(9, 3.2)); plt.plot(판점수, alpha=.2, c='#9aa3ad', label='한 판 점수')
    if len(판점수) >= 50: plt.plot(range(49, len(판점수)), np.convolve(판점수, np.ones(50) / 50, 'valid'), lw=2.5, c='#00c832', label='최근 50판 평균')
    if 기준: plt.axhline(기준, ls='--', c='#ffb547', label=f'착지 +{기준}')
    plt.xlabel('판'); plt.ylabel('점수'); plt.title(제목); plt.legend(); plt.tight_layout(); plt.show()
print('✅ PPO학습() 준비 — ① 정책 두뇌 ② 한 판 통째로 ③ 어드밴티지 ④ clip')

## CELL 09 · 🚗 버튼 4개로 — 뒤집힌 로드스터 착지

5강 DQN 과 **같은 세상 · 같은 버튼 4개**. 100만 걸음을 학습합니다 (코랩 CPU 로 약 2 ~ 4분).
`시험` 은 학습 중 처음 보는 20판의 착지율, 🏅 는 그때까지 가장 좋았던 순간입니다. 마지막엔 **학습이 끝난 그 두뇌**로 처음 보는 50판을 시험합니다.

In [ ]:
버튼두뇌, 버튼점수, 버튼시험, _ = PPO학습(로드스터, 1_000_000, 탐험보너스=0.01, 씨앗=0, 보고=100_000,
                                    평가=lambda b: 착지율(b, 로드스터, 판=20))
print(f'\n✅ 마지막 두뇌 · 처음 보는 50판 착지율 : {착지율(버튼두뇌, 로드스터, 50):.0f}%   (5강 DQN 마지막 두뇌 0%)')
곡선(버튼점수, 'PPO · 버튼 4개 — 뒤집힌 로드스터', 100)

def 영상(뇌, 만들기, 씨앗, 제목, 파일, 색=(0, 255, 65)):
    e = 만들기(); o, _ = e.reset(seed=씨앗); 화면들 = []; 끝 = 잘림 = False
    while not (끝 or 잘림):
        with torch.no_grad(): d = 뇌.분포(torch.as_tensor(o, dtype=torch.float32))
        if 뇌.연속: a = np.clip(d.mean.numpy(), e.action_space.low, e.action_space.high); 화면들.append(로드스터그림(e, None, 제목, 색, 세기=a))
        else: a = int(d.probs.argmax()); 화면들.append(로드스터그림(e, a, 제목, 색))
        o, r, 끝, 잘림, 정보 = e.step(a)
    화면들 += [로드스터그림(e, None, f"{제목} → {정보['결과']}", 색)] * 25; 영상보기(화면들, 파일); return 정보['결과']
print('PPO · 버튼 →', 영상(버튼두뇌, 로드스터, 7, 'PPO · 버튼 4개', 'roadster_ppo_button.mp4'))

## CELL 10 · 🕹 조이스틱으로 — 세기를 직접 고른다

5강 실습 2 에서 말한 **continuous 액션** 입니다. 축구 게임에서 버튼을 살짝 · 세게 누르듯, 추진기를 0 ~ 100% 사이 **아무 세기로나** 켭니다.
DQN 은 「몇 개 중 하나 고르기」 만 할 수 있어 이 세상에 그대로 붙일 수 없습니다. PPO 는 정책 머리만 「평균 · 흔들림」 으로 바꾸면 됩니다 — **같은 함수 그대로**.

In [ ]:
조이두뇌, 조이점수, 조이시험, _ = PPO학습(로드스터연속, 1_000_000, 씨앗=0, 보고=100_000,
                                    평가=lambda b: 착지율(b, 로드스터연속, 판=20))
print(f'\n✅ 마지막 두뇌 · 처음 보는 50판 착지율 : {착지율(조이두뇌, 로드스터연속, 50):.0f}%')
print('   배운 흔들림(표준편차) :', np.round(조이두뇌.흔들림.exp().detach().numpy(), 3), '← 처음 0.61 에서 줄었다 = 세기가 정확해졌다')
곡선(조이점수, 'PPO · 조이스틱 — 뒤집힌 로드스터', 100)
print('PPO · 조이스틱 →', 영상(조이두뇌, 로드스터연속, 7, 'PPO · 조이스틱', 'roadster_ppo_joystick.mp4'))

## CELL 11 · 🔍 두뇌 속 들여다보기 — 매 순간의 확률

다 배운 버튼 두뇌가 매 순간 네 버튼에 매기는 **확률**을 오른쪽 막대로 붙였습니다.
뒤집혀 있을 땐 냉가스(회전) 확률이 높고, 바로 서면 아래 추진기 확률이 올라가는 걸 보세요. 4강의 π(a|s) 가 실제로 움직이는 모습입니다.

In [ ]:
def 확률막대화면(화면, p, a):
    im = Image.fromarray(화면); W, H = im.size; 판 = Image.new('RGB', (W + 230, H), (7, 8, 10)); 판.paste(im, (0, 0)); d = ImageDraw.Draw(판)
    d.text((W + 14, 12), '두뇌의 확률 π(a|s)', fill=(191, 239, 255), font=글꼴(16))
    for k, (이름, q) in enumerate(zip(버튼, p)):
        y = 50 + k * 70; d.text((W + 14, y), 이름, fill=(220, 225, 230), font=글꼴(14))
        d.rectangle([W + 14, y + 22, W + 14 + int(190 * q), y + 40], fill=(0, 255, 65) if k == a else (90, 100, 112))
        d.text((W + 14, y + 44), f'{q*100:4.1f}%', fill=(200, 210, 220), font=글꼴(12))
    return np.array(판)
e = 로드스터(); o, _ = e.reset(seed=11); 화면들 = []; 끝 = 잘림 = False
while not (끝 or 잘림):
    with torch.no_grad(): p = 버튼두뇌.분포(torch.as_tensor(o, dtype=torch.float32)).probs.numpy()
    a = int(p.argmax()); 화면들.append(확률막대화면(로드스터그림(e, a, 'PPO · 매 순간의 확률'), p, a)); o, r, 끝, 잘림, 정보 = e.step(a)
영상보기(화면들 + [화면들[-1]] * 20, 'roadster_ppo_probs.mp4', 너비=640); print('결과 :', 정보['결과'])

## CELL 12 · DQN vs PPO — 같은 세상, 씨앗 5개

한 번 잘된 게 운인지 보려고 **씨앗(시작 운) 5개**로 다시 돌렸습니다. 모두 100만 걸음, 학습이 끝난 **마지막 두뇌**로 처음 보는 50판 착지율.

| 두뇌 | 씨앗 0 | 1 | 2 | 3 | 4 | 평균 |
|---|---|---|---|---|---|---|
| DQN · 버튼 4개 (5강, 80만 걸음) | 0% | 0% | 0% | 0% | 0% | **0%** |
| PPO · 버튼 4개 | 98% | 78% | 92% | 90% | 84% | **88%** |
| PPO · 조이스틱 | 100% | 84% | 94% | 98% | 100% | **95%** |

<sub>맥 M5 CPU 로 한 번에 39 ~ 59초 (100만 걸음). 코랩 CPU 는 2 ~ 4배쯤 걸립니다.</sub>

**DQN 은 반짝 배웠다 잊었고, PPO 는 끝까지 붙잡았습니다.** 이유는 오늘 배운 세 가지입니다.
1. **확률로 고르기** — 「돌고 → 멈추고 → 줄이는」 긴 순서를 확률이 자연스럽게 이어 갑니다. ε 동전처럼 걸음마다 따로 흩어지지 않습니다.
2. **방금 해 본 판으로만** — 오래된 추락 기억에 착지 경험이 묻히지 않습니다.
3. **clip** — 한 번에 조금씩만 고쳐서, 배운 착지를 갑자기 버리지 않습니다.

> 그래서 요즘 로봇 · 휴머노이드 · 게임 AI 의 기본기가 PPO 입니다. 다만 DQN 이 쓸모없는 게 아닙니다 — 버튼이 몇 개뿐이고 경험이 귀한 세상에선 기억 상자로 여러 번 복습하는 DQN 이 더 알뜰합니다. 세상에 맞는 두뇌를 고르는 눈, 그게 오늘의 수확입니다.

## CELL 13 · 🏆 도전 — 숫자를 바꿔 착지율을 올려라

아래 숫자를 바꾸고 실행해 보세요. 같은 씨앗이면 같은 결과가 나오니, **하나씩만** 바꿔야 무엇이 효과였는지 압니다.

| 숫자 | 뜻 | 해 볼 것 |
|---|---|---|
| `자르기` | clip 폭 (0.2 = ±20%) | 0.05 (너무 조심) · 0.5 (너무 과감) |
| `세상수` | 동시에 돌리는 세상 | 2 · 16 |
| `학습률` | 한 번에 고치는 크기 | 1e-3 · 1e-4 |
| `총걸음` | 얼마나 오래 | 50만 · 200만 |

In [ ]:
총걸음, 자르기, 세상수, 학습률, 씨앗 = 500_000, 0.2, 8, 3e-4, 0     # ✏️ 바꿔 보세요
t0 = time.time()
도전두뇌, 도전점수, _, _ = PPO학습(로드스터연속, 총걸음, 세상수=세상수, 자르기=자르기, 학습률=학습률, 씨앗=씨앗, 보고=100_000)
print(f'\n🏆 자르기 {자르기} · 세상 {세상수}개 · 학습률 {학습률} · {총걸음:,}걸음 ({time.time()-t0:.0f}초)')
print(f'   처음 보는 50판 착지율 : {착지율(도전두뇌, 로드스터연속, 50):.0f}%')

## CELL 14 · 🤗 허깅페이스 준비 — 토큰 넣기

5강과 같습니다. 코랩 왼쪽 **🔑 (Secrets)** 에 `HF_TOKEN` (Write 토큰)을 넣거나, 아래 칸을 실행해 나오는 입력창에 붙여넣으세요.

> ⚠️ 토큰은 비밀번호와 같습니다. 화면에 보이게 붙여넣거나 다른 사람에게 보내지 마세요.

In [ ]:
from huggingface_hub import login, HfApi
토큰 = None
try:
    from google.colab import userdata; 토큰 = userdata.get('HF_TOKEN')
except Exception: pass
if not 토큰: 토큰 = os.environ.get('HF_TOKEN')
if not 토큰:
    try:
        from getpass import getpass; 토큰 = getpass('허깅페이스 Write 토큰을 붙여넣으세요 (화면에 안 보입니다): ').strip() or None
    except Exception: 토큰 = None
if 토큰:
    login(token=토큰, add_to_git_credential=False); 나 = HfApi().whoami()['name']; print(f'✅ 로그인 완료 — 내 허깅페이스 이름: {나}')
else:
    나 = None; print('⏭️ 토큰이 없어 업로드는 건너뜁니다. (학습한 두뇌는 그대로 있습니다)')

## CELL 15 · 내 이름으로 PPO 두뇌 올리기 — 이름은 알아서

세 칸만 채우고 실행하세요. 올라가는 이름은 **`connectailab-rl-이름-버전-roadster-ppo`** — 5강 루나랜더 모델과 같은 규칙이라 내 모델들이 한 줄로 모입니다.

In [ ]:
이름 = ""                                           #@param {type:"string"}
버전 = "v1"                                          #@param {type:"string"}
나를_알리는_글 = """
"""
import re, datetime, shutil
def 저장소이름(이름, 버전, 종류):
    깨끗 = lambda s: re.sub(r'[^a-z0-9.]+', '-', str(s).lower()).strip('-.')
    n, v = 깨끗(이름), 깨끗(버전) or 'v1'
    if not n: raise ValueError('이름은 영문 · 숫자로 적어 주세요 (예: jay, minji, team3)')
    return f'connectailab-rl-{n}-{v}-{깨끗(종류)}'

올린주소 = None
if 나:
    저장소 = 저장소이름(이름.strip() or 나, 버전, 'roadster-ppo'); 올린주소 = f'{나}/{저장소}'; 폴더 = f'hf_{저장소}'; os.makedirs(폴더, exist_ok=True)
    점수 = 착지율(조이두뇌, 로드스터연속, 50)
    torch.save(조이두뇌.state_dict(), f'{폴더}/model.pt'); shutil.copy('로드스터_시뮬레이터.py', f'{폴더}/로드스터_시뮬레이터.py')
    if os.path.exists('roadster_ppo_joystick.mp4'): shutil.copy('roadster_ppo_joystick.mp4', f'{폴더}/replay.mp4')
    소개 = '\n'.join(l.rstrip() for l in 나를_알리는_글.strip().splitlines())[:3000]
    설정 = {'algo': 'PPO', 'steps': 1_000_000, 'envs': 8, 'clip': 0.2, 'gamma': 0.99, 'gae_lambda': 0.95, 'lr': 3e-4}
    json.dump({'format': 'connectailab/1', 'repo': 올린주소, 'kind': 'ppo', 'env': 'Inverted Roadster (continuous)', 'algo': 'ppo',
               'author': {'about': 소개} if 소개 else None, 'config': 설정, 'test': {'landing_rate_50': 점수},
               'course': 'AI CITY BUILDERS 강화학습 AI 6강', 'date': str(datetime.date.today())}, open(f'{폴더}/connectailab.json', 'w'), ensure_ascii=False, indent=1)
    만든사람 = f"## 만든 사람\n\n{소개}\n\n<sub>CONNECT AI LAB 공동 연구 참여 모델</sub>\n\n" if 소개 else ''
    카드 = f"""---
tags: [reinforcement-learning, ppo, continuous-control, connect-ai-lab, connectailab]
library_name: pytorch
---
# {저장소}

{만든사람}뒤집힌 채로 떨어지는 로드스터를 추진기 **세기(0 ~ 1)와 회전(−1 ~ 1)** 으로 돌려 바퀴부터 착지시키는 **PPO** 두뇌입니다.

| 항목 | 내용 |
|---|---|
| 세상 | 뒤집힌 로드스터 (`로드스터_시뮬레이터.py` 의 `로드스터연속`, Gymnasium) |
| 상태 | 위치 x · y, 속도 x · y, 기울기 sin · cos, 회전 속도 (7개) |
| 행동 | [아래 추진기 세기, 회전 세기] — 연속 2개 |
| 결과 | **처음 보는 50판 착지율 {점수:.0f}%** |
| 학습 | {', '.join(f'{k} {v}' for k, v in 설정.items())} |

AI CITY BUILDERS 강화학습 AI 6강 · CONNECT AI LAB
"""
    open(f'{폴더}/README.md', 'w').write(카드)
    HfApi().create_repo(저장소, exist_ok=True); HfApi().upload_folder(folder_path=폴더, repo_id=올린주소)
    print(f'🎉 올렸습니다 → https://huggingface.co/{올린주소}  (착지율 {점수:.0f}%)')
else:
    print('⏭️ 로그인하지 않아 건너뜁니다.')

## CELL 16 · 내려받아 다시 쓰기 — 진짜 공개 모델인지 확인

In [ ]:
if 올린주소:
    from huggingface_hub import hf_hub_download
    받은 = 정책두뇌(7, 2, 연속=True); 받은.load_state_dict(torch.load(hf_hub_download(올린주소, 'model.pt', force_download=True), map_location='cpu'))
    print(f'⬇️ {올린주소} 에서 받은 두뇌 → 처음 보는 20판 착지율 {착지율(받은, 로드스터연속, 20):.0f}%')
else:
    print('⏭️ 업로드를 건너뛰어 이 칸도 건너뜁니다.')

## CELL 17 · 👉 7강 예고 — 관절이 여럿인 로봇

오늘 조이스틱은 숫자 **2개**(추진기 세기 · 회전)였습니다. 로봇은 어떨까요?

| 로봇 | 동시에 정하는 세기 |
|---|---|
| 로드스터 (오늘) | 2개 |
| Hopper (한 발) | 3개 |
| Walker2d (두 발) | 6개 |
| Humanoid (사람 모양) | 17개 |

DQN 으로 버튼을 만들려면 세기를 잘게 잘라야 하고, 관절이 늘면 버튼 수가 폭발합니다(5강 부록 E). PPO 는 **평균 · 흔들림만 관절 수만큼 늘리면** 끝입니다.
**7강에서는 MuJoCo 로 두 발 로봇을 걷게 합니다.**

## 정리 — 6강에서 한 것

| | DQN (5강) | PPO (6강) |
|---|---|---|
| 두뇌가 배우는 것 | 행동마다 점수 Q | **행동을 고를 확률 π** (+ 가치 V) |
| 고르는 법 | 가장 큰 Q · 탐험은 ε 동전 | **확률대로 굴리기** = 탐험 |
| 경험 | 기억 상자에 쌓아 두고 복습 | **방금 해 본 판으로 배우고 버림** · 여러 세상 동시에 |
| 고치는 법 | 정답(보상 + γ·다음 Q)에 맞추기 | 잘한 행동 확률 ↑ 못한 행동 ↓ · **clip ±20%** |
| 행동 | 버튼만 | 버튼 · **조이스틱(세기)** 둘 다 |
| 뒤집힌 로드스터 | 마지막 두뇌 0% | **버튼 평균 88% · 조이스틱 평균 95%** (씨앗 5개) |

- **PPO = 정책 두뇌 + ② 한 판 통째로 + ③ 어드밴티지(γ · 4강) + ④ clip** — 이름 그대로 「가까이에서만 정책을 고친다」.
- 조이스틱 PPO 두뇌를 허깅페이스에 올렸습니다 — 루나랜더(5강)에 이어 두 번째 공개 모델입니다.
- 👉 **7강** — 관절 6개, 두 발로 걷는 로봇.

> 🔒 이 노트북과 코드는 Connect AI LAB 이 만들었습니다. 공부에는 마음껏 쓰시고, 다른 강의를 만드는 데는 쓰지 말아 주세요.